In [ ]:
import hashlib
import json
import math
import os
import random
from collections import defaultdict
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import mido
import numpy as np
from tqdm.auto import tqdm

try:
    import h5py
    H5PY_AVAILABLE = True
    print("Ase bhai")
except Exception:
    h5py = None
    H5PY_AVAILABLE = False

: 

In [ ]:
@dataclass
class PreprocessConfig:
    project_root: Path = Path(os.environ.get("PROJECT_ROOT", Path.cwd())).resolve()
    if not (project_root / "data").exists() and (project_root.parent / "data").exists():
        project_root = project_root.parent
    raw_midi_dir: Path = project_root / "data" / "raw_midi"
    lmd_h5_dir: Path = project_root / "data" / "dataset"
    output_dir: Path = project_root / "data" / "processed"

    use_raw_midi: bool = True
    use_lmd_h5: bool = False

    max_files: Optional[int] = None
    random_seed: int = 42

    fs: int = 16
    max_roll_steps: int = 2048
    min_notes_per_track: int = 8

    duration_bins: Tuple[float, ...] = (0.1, 0.25, 0.5, 1.0, 2.0, 4.0)
    velocity_bins: Tuple[int, ...] = (16, 32, 48, 64, 80, 96, 112, 127)
    # Time-shift bins encode delta between consecutive note start times.
    time_shift_bins: Tuple[float, ...] = (0.02, 0.05, 0.1, 0.2, 0.5, 1.0, 2.0, 4.0)

    test_size: float = 0.10
    val_size_from_train: float = 0.10

    chunk_size: int = 256
    checkpoint_every: int = 500
    merge_final_npz: bool = False
    show_progress: bool = True
    save_compressed: bool = True


config = PreprocessConfig()
config.output_dir.mkdir(parents=True, exist_ok=True)
print(json.dumps({k: str(v) if isinstance(v, Path) else v for k, v in asdict(config).items()}, indent=2))

In [ ]:
quick_validate_max_files: Optional[int] = None
if quick_validate_max_files is not None:
    config.max_files = quick_validate_max_files
    print(f"Quick validation enabled: max_files={config.max_files}")

In [ ]:
MIDI_EXTENSIONS = {".mid", ".midi", ".MID", ".MIDI"}


def discover_files(root: Path, allowed_suffixes: Optional[Sequence[str]] = None, limit: Optional[int] = None) -> List[Path]:
    if not root.exists():
        return []
    if allowed_suffixes is None:
        files = [p for p in root.rglob("*") if p.is_file()]
    else:
        suffixes = set(allowed_suffixes)
        files = [p for p in root.rglob("*") if p.is_file() and p.suffix in suffixes]
    files.sort()
    return files[:limit] if limit is not None else files


def find_input_files(cfg: PreprocessConfig) -> Dict[str, List[Path]]:
    inputs: Dict[str, List[Path]] = {"raw_midi": [], "lmd_h5": []}
    if cfg.use_raw_midi:
        inputs["raw_midi"] = discover_files(cfg.raw_midi_dir, list(MIDI_EXTENSIONS), cfg.max_files)
    if cfg.use_lmd_h5:
        inputs["lmd_h5"] = discover_files(cfg.lmd_h5_dir, [".h5", ".hdf5"], cfg.max_files)
    return inputs


def inspect_first_h5_schema(input_index: Dict[str, List[Path]], max_items: int = 20) -> None:
    if not H5PY_AVAILABLE:
        print("h5py not available.")
        return
    h5_files = input_index.get("lmd_h5", [])
    if not h5_files:
        print("No H5 files discovered.")
        return
    target = h5_files[0]
    print("Inspecting:", target)
    with h5py.File(target, "r") as f:
        items = []
        f.visititems(lambda name, obj: items.append((name, type(obj).__name__, getattr(obj, "shape", None))))
        for i, (name, typ, shape) in enumerate(items[:max_items]):
            print(f"{i+1:02d}. {name} | {typ} | shape={shape}")


def parse_raw_midi_file(path: Path) -> Optional[List[Tuple[int, float, float, int]]]:
    try:
        midi = mido.MidiFile(str(path))
    except Exception:
        return None

    current_time = 0.0
    active: Dict[Tuple[int, int], List[Tuple[float, int]]] = defaultdict(list)
    notes: List[Tuple[int, float, float, int]] = []

    try:
        for msg in midi:
            current_time += float(msg.time)
            if msg.type == "note_on" and getattr(msg, "velocity", 0) > 0:
                key = (int(msg.note), int(getattr(msg, "channel", 0)))
                active[key].append((current_time, int(msg.velocity)))
            elif msg.type == "note_off" or (msg.type == "note_on" and getattr(msg, "velocity", 0) == 0):
                key = (int(msg.note), int(getattr(msg, "channel", 0)))
                if active[key]:
                    start, velocity = active[key].pop(0)
                    if current_time > start:
                        notes.append((int(msg.note), float(start), float(current_time), int(velocity)))
    except Exception:
        return None

    notes.sort(key=lambda x: (x[1], x[0]))
    return notes if notes else None


def parse_lmd_h5_file(path: Path) -> Optional[List[Tuple[int, float, float, int]]]:
    return None


def quantize_duration(duration: float, bins: Sequence[float]) -> int:
    for i, b in enumerate(bins):
        if duration <= b:
            return i
    return len(bins)


def quantize_velocity(velocity: int, bins: Sequence[int]) -> int:
    for i, b in enumerate(bins):
        if velocity <= b:
            return i
    return len(bins)


def quantize_time_shift(delta_t: float, bins: Sequence[float]) -> int:
    for i, b in enumerate(bins):
        if delta_t <= b:
            return i
    return len(bins)


def notes_to_tokens(
    notes: Sequence[Tuple[int, float, float, int]],
    duration_bins: Sequence[float],
    velocity_bins: Sequence[int],
    time_shift_bins: Sequence[float],
) -> List[int]:
    # Event vocabulary: [note events] + [time-shift events]
    # note_token = pitch + 128*d_bin + 128*(len(duration_bins)+1)*v_bin
    # time_shift_token = note_token_count + t_bin
    tokens: List[int] = []
    if not notes:
        return tokens

    note_token_count = 128 * (len(duration_bins) + 1) * (len(velocity_bins) + 1)
    prev_start = 0.0

    for pitch, start, end, velocity in notes:
        start = float(start)
        delta_t = max(0.0, start - prev_start)
        if delta_t > 0.0:
            t_bin = quantize_time_shift(delta_t, time_shift_bins)
            tokens.append(note_token_count + t_bin)

        duration = max(0.01, float(end) - float(start))
        d_bin = quantize_duration(duration, duration_bins)
        v_bin = quantize_velocity(int(velocity), velocity_bins)
        token = int(pitch) + 128 * d_bin + 128 * (len(duration_bins) + 1) * v_bin
        tokens.append(token)

        prev_start = start
    return tokens


def notes_to_pianoroll(
    notes: Sequence[Tuple[int, float, float, int]],
    fs: int,
    max_steps: int,
) -> np.ndarray:
    if not notes:
        return np.zeros((max_steps, 128), dtype=np.float32)
    max_end = max(end for _, _, end, _ in notes)
    total_steps = min(max_steps, max(1, int(math.ceil(max_end * fs))))
    roll = np.zeros((total_steps, 128), dtype=np.float32)
    for pitch, start, end, velocity in notes:
        start_idx = max(0, int(math.floor(start * fs)))
        end_idx = min(total_steps, max(start_idx + 1, int(math.ceil(end * fs))))
        if start_idx >= total_steps:
            continue
        value = float(velocity) / 127.0
        roll[start_idx:end_idx, pitch] = np.maximum(roll[start_idx:end_idx, pitch], value)
    if total_steps < max_steps:
        padded = np.zeros((max_steps, 128), dtype=np.float32)
        padded[:total_steps] = roll
        return padded
    return roll


def split_name_for_example(path_str: str, cfg: PreprocessConfig) -> str:
    stable_key = f"{path_str}|{cfg.random_seed}".encode("utf-8")
    stable_int = int.from_bytes(hashlib.sha1(stable_key).digest()[:8], byteorder="big", signed=False)
    rnd = random.Random(stable_int)
    r = rnd.random()
    if r < cfg.test_size:
        return "test"
    val_threshold = cfg.test_size + (1.0 - cfg.test_size) * cfg.val_size_from_train
    if r < val_threshold:
        return "val"
    return "train"


def flush_chunk(
    cfg: PreprocessConfig,
    split_name: str,
    chunk_id: int,
    rows: List[dict],
    manifests: Dict[str, List[str]],
) -> None:
    if not rows:
        return
    split_dir = cfg.output_dir / split_name
    split_dir.mkdir(parents=True, exist_ok=True)
    out_path = split_dir / f"chunk_{chunk_id:06d}.npz"
    save_fn = np.savez_compressed if cfg.save_compressed else np.savez
    save_fn(
        out_path,
        token_sequences=np.array([r["tokens"] for r in rows], dtype=object),
        piano_rolls=np.stack([r["pianoroll"] for r in rows], axis=0).astype(np.float32),
        relative_paths=np.array([r["relative_path"] for r in rows], dtype=object),
        source_types=np.array([r["source_type"] for r in rows], dtype=object),
    )
    manifests[split_name].append(str(out_path.relative_to(cfg.output_dir)))


def process_streaming(cfg: PreprocessConfig, input_index: Dict[str, List[Path]]) -> dict:
    random.seed(cfg.random_seed)
    sources: List[Tuple[str, Path]] = []
    sources.extend(("raw_midi", p) for p in input_index.get("raw_midi", []))
    sources.extend(("lmd_h5", p) for p in input_index.get("lmd_h5", []))

    stats = {
        "seen": 0,
        "parsed_ok": 0,
        "parse_failed": 0,
        "skipped_too_few_notes": 0,
        "raw_midi_used": 0,
        "lmd_h5_used": 0,
        "split_counts": {"train": 0, "val": 0, "test": 0},
    }
    buffers: Dict[str, List[dict]] = {"train": [], "val": [], "test": []}
    chunk_ids = {"train": 0, "val": 0, "test": 0}
    manifests: Dict[str, List[str]] = {"train": [], "val": [], "test": []}

    source_iter: Iterable[Tuple[str, Path]] = sources
    if cfg.show_progress:
        source_iter = tqdm(sources, desc="Preprocessing", unit="file")

    for source_type, path in source_iter:
        stats["seen"] += 1
        notes = parse_raw_midi_file(path) if source_type == "raw_midi" else parse_lmd_h5_file(path)
        if notes is None:
            stats["parse_failed"] += 1
            continue
        if len(notes) < cfg.min_notes_per_track:
            stats["skipped_too_few_notes"] += 1
            continue

        row = {
            "source_type": source_type,
            "relative_path": str(path.relative_to(cfg.project_root)),
            "tokens": notes_to_tokens(notes, cfg.duration_bins, cfg.velocity_bins, cfg.time_shift_bins),
            "pianoroll": notes_to_pianoroll(notes, cfg.fs, cfg.max_roll_steps),
        }
        split_name = split_name_for_example(row["relative_path"], cfg)
        buffers[split_name].append(row)
        stats["split_counts"][split_name] += 1
        stats["parsed_ok"] += 1
        stats["raw_midi_used"] += int(source_type == "raw_midi")
        stats["lmd_h5_used"] += int(source_type == "lmd_h5")

        if len(buffers[split_name]) >= cfg.chunk_size:
            flush_chunk(cfg, split_name, chunk_ids[split_name], buffers[split_name], manifests)
            print(f"Saved chunk {chunk_ids[split_name]} for {split_name} ({len(buffers[split_name])} rows)")
            chunk_ids[split_name] += 1
            buffers[split_name] = []

        if stats["seen"] % cfg.checkpoint_every == 0:
            with open(cfg.output_dir / "preprocessing_checkpoint.json", "w", encoding="utf-8") as f:
                json.dump({"stats": stats, "chunk_counts": {k: len(v) for k, v in manifests.items()}}, f, indent=2)

    for split_name in ("train", "val", "test"):
        if buffers[split_name]:
            flush_chunk(cfg, split_name, chunk_ids[split_name], buffers[split_name], manifests)
            print(f"Saved final chunk {chunk_ids[split_name]} for {split_name} ({len(buffers[split_name])} rows)")

    metadata = {
        "config": {k: str(v) if isinstance(v, Path) else v for k, v in asdict(cfg).items()},
        "stats": stats,
        "chunk_manifests": manifests,
    }
    with open(cfg.output_dir / "preprocessing_metadata.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)
    return metadata


def merge_split_chunks(cfg: PreprocessConfig, split_name: str, manifest: List[str]) -> Optional[Path]:
    if not manifest:
        return None
    all_tokens: List[list] = []
    all_paths: List[str] = []
    all_sources: List[str] = []
    all_rolls: List[np.ndarray] = []
    chunk_iter: Iterable[str] = manifest
    if cfg.show_progress:
        chunk_iter = tqdm(manifest, desc=f"Merging {split_name}", unit="chunk")
    for rel in chunk_iter:
        with np.load(cfg.output_dir / rel, allow_pickle=True) as data:
            all_tokens.extend(data["token_sequences"].tolist())
            all_paths.extend(data["relative_paths"].tolist())
            all_sources.extend(data["source_types"].tolist())
            all_rolls.append(data["piano_rolls"].astype(np.float32))
    out_path = cfg.output_dir / f"{split_name}_dataset.npz"
    save_fn = np.savez_compressed if cfg.save_compressed else np.savez
    save_fn(
        out_path,
        token_sequences=np.array(all_tokens, dtype=object),
        piano_rolls=np.concatenate(all_rolls, axis=0) if all_rolls else np.zeros((0, cfg.max_roll_steps, 128), dtype=np.float32),
        relative_paths=np.array(all_paths, dtype=object),
        source_types=np.array(all_sources, dtype=object),
    )
    return out_path


def validate_architecture_and_outputs(cfg: PreprocessConfig) -> None:
    expected_notebooks = [
        cfg.project_root / "notebooks" / "0_preprocessing.ipynb",
        cfg.project_root / "notebooks" / "task1_lstm_ae.ipynb",
        cfg.project_root / "notebooks" / "task2_vae.ipynb",
        cfg.project_root / "notebooks" / "task3_transformer.ipynb",
        cfg.project_root / "notebooks" / "task4_rlhf.ipynb",
    ]
    project_details_pdf = cfg.project_root / "data" / "Project_Details_of_Neural_Network_Spring_2026.pdf"
    print("Architecture checks:")
    print("- Notebook pipeline exists:", all(p.exists() for p in expected_notebooks))
    print("- Input/output dirs exist:", all(p.exists() for p in [cfg.raw_midi_dir, cfg.output_dir]))
    print("- Project details PDF exists:", project_details_pdf.exists())

    metadata_path = cfg.output_dir / "preprocessing_metadata.json"
    if not metadata_path.exists():
        print("No preprocessing metadata found.")
        return
    with open(metadata_path, "r", encoding="utf-8") as f:
        meta = json.load(f)
    manifests = meta.get("chunk_manifests", {})
    print("\nPreprocessing stats:", meta.get("stats", {}))
    print("Chunk counts:", {k: len(v) for k, v in manifests.items()})
    print("\nOutput path checks:")
    for split_name in ("train", "val", "test"):
        split_dir = cfg.output_dir / split_name
        chunk_paths = [cfg.output_dir / rel for rel in manifests.get(split_name, [])]
        print(split_name, "split_dir_exists:", split_dir.exists(), "chunks_exist:", all(p.exists() for p in chunk_paths))
    print("\nProject-fit checks:")
    print("- Uses time-aware note events with delta-time (time-shift tokens).")
    print("- Produces token + piano-roll features and train/val/test splits.")
    print("- Saves outputs under data/processed for downstream notebooks.")

In [ ]:
inputs = find_input_files(config)
print("Input counts:", {k: len(v) for k, v in inputs.items()})

if config.use_lmd_h5:
    inspect_first_h5_schema(inputs)

In [ ]:
metadata = process_streaming(config, inputs)
print("parsed_ok:", metadata["stats"]["parsed_ok"])
print("split_counts:", metadata["stats"]["split_counts"])

In [ ]:
if config.merge_final_npz:
    with open(config.output_dir / "preprocessing_metadata.json", "r", encoding="utf-8") as f:
        meta = json.load(f)
    for split_name in ("train", "val", "test"):
        merged = merge_split_chunks(config, split_name, meta["chunk_manifests"][split_name])
        print(split_name, "merged:", merged if merged else "no chunks")
else:
    print("Skipping final merge (merge_final_npz=False).")
    print("Chunked outputs are ready under data/processed/{train,val,test}/")

In [ ]:
validate_architecture_and_outputs(config)

Task 1

In [ ]:
from __future__ import annotations

import json
import math
import os
import random
from dataclasses import dataclass
from pathlib import Path
from typing import List, Tuple

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

In [ ]:
# Why: keep reusable data/model utilities in one place for clean, repeatable runs.

@dataclass
class Task1Config:
    project_root: Path = Path(os.environ.get("PROJECT_ROOT", Path.cwd())).resolve()
    if not (project_root / "data").exists() and (project_root.parent / "data").exists():
        project_root = project_root.parent
    processed_dir: Path = project_root / "data" / "processed"
    metadata_file: Path = processed_dir / "preprocessing_metadata.json"
    artifacts_dir: Path = project_root / "artifacts" / "task1_lstm_ae"

    seq_len: int = 256
    batch_size: int = 16
    num_workers: int = 0

    embedding_dim: int = 128
    hidden_dim: int = 256
    num_layers: int = 2
    dropout: float = 0.2

    lr: float = 1e-3
    weight_decay: float = 1e-5
    epochs: int = 5
    grad_clip: float = 1.0

    # Set to None for full-dataset training.
    max_train_examples: int | None = 40000
    max_val_examples: int | None = 5000
    max_test_examples: int | None = 5000


def load_split_token_sequences(
    split: str,
    max_examples: int | None,
    manifest_map: dict,
    processed_dir: Path,
    vocab_size: int,
) -> List[List[int]]:
    manifest = manifest_map.get(split, [])
    sequences: List[List[int]] = []

    for rel in tqdm(manifest, desc=f"Loading {split}", unit="chunk"):
        chunk_path = processed_dir / rel
        if not chunk_path.exists():
            continue
        with np.load(chunk_path, allow_pickle=True) as data:
            raw = data["token_sequences"].tolist()
            for seq in raw:
                if not seq:
                    continue
                shifted = [min(int(t) + 1, vocab_size - 1) for t in seq]
                sequences.append(shifted)
                if max_examples is not None and len(sequences) >= max_examples:
                    return sequences
    return sequences

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

cfg = Task1Config()
cfg.artifacts_dir.mkdir(parents=True, exist_ok=True)
assert cfg.metadata_file.exists(), f"Missing metadata: {cfg.metadata_file}"
print("Using metadata:", cfg.metadata_file)

with open(cfg.metadata_file, "r", encoding="utf-8") as f:
    meta = json.load(f)

duration_bins = tuple(meta["config"]["duration_bins"])
velocity_bins = tuple(meta["config"]["velocity_bins"])
time_shift_bins = tuple(meta["config"].get("time_shift_bins", []))

note_token_space = 128 * (len(duration_bins) + 1) * (len(velocity_bins) + 1)
time_shift_space = (len(time_shift_bins) + 1) if len(time_shift_bins) > 0 else 0
token_space = note_token_space + time_shift_space
pad_id = 0
vocab_size = token_space + 1
print("token_space:", token_space, "vocab_size:", vocab_size)

train_sequences = load_split_token_sequences(
    "train",
    cfg.max_train_examples,
    meta["chunk_manifests"],
    cfg.processed_dir,
    vocab_size,
    )
val_sequences = load_split_token_sequences(
    "val",
    cfg.max_val_examples,
    meta["chunk_manifests"],
    cfg.processed_dir,
    vocab_size,
    )
test_sequences = load_split_token_sequences(
    "test",
    cfg.max_test_examples,
    meta["chunk_manifests"],
    cfg.processed_dir,
    vocab_size,
    )

print({
    "train": len(train_sequences),
    "val": len(val_sequences),
    "test": len(test_sequences),
})

In [ ]:
train_ds = TokenSequenceDataset(train_sequences, cfg.seq_len, pad_id)
val_ds = TokenSequenceDataset(val_sequences, cfg.seq_len, pad_id)
test_ds = TokenSequenceDataset(test_sequences, cfg.seq_len, pad_id)

train_loader = DataLoader(
    train_ds,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    val_ds,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    test_ds,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=torch.cuda.is_available(),
)

print("Batches:", {
    "train": len(train_loader),
    "val": len(val_loader),
    "test": len(test_loader),
})

In [ ]:
model = LSTMAutoencoder(
    vocab_size=vocab_size,
    embedding_dim=cfg.embedding_dim,
    hidden_dim=cfg.hidden_dim,
    num_layers=cfg.num_layers,
    dropout=cfg.dropout,
    pad_id=pad_id,
).to(DEVICE)

criterion = nn.CrossEntropyLoss(ignore_index=pad_id)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=cfg.lr,
    weight_decay=cfg.weight_decay,
    betas=(0.9, 0.999),
)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")

best_val_loss = float("inf")
history = []

for epoch in range(1, cfg.epochs + 1):
    train_loss, train_acc = run_epoch(
        model,
        train_loader,
        criterion,
        pad_id,
        DEVICE,
        train=True,
        optimizer=optimizer,
        grad_clip=cfg.grad_clip,
        desc=f"train e{epoch}",
    )
    val_loss, val_acc = run_epoch(
        model,
        val_loader,
        criterion,
        pad_id,
        DEVICE,
        train=False,
        desc=f"val e{epoch}",
    )

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "train_acc": train_acc,
        "val_acc": val_acc,
    }
    history.append(row)
    print(row)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        ckpt_path = cfg.artifacts_dir / "best_lstm_autoencoder.pt"
        torch.save(
            {
                "model_state": model.state_dict(),
                "config": cfg.__dict__,
                "vocab_size": vocab_size,
                "pad_id": pad_id,
                "history": history,
            },
            ckpt_path,
        )
        print(f"Saved best checkpoint -> {ckpt_path}")

In [ ]:
# Load best checkpoint for final evaluation
ckpt_path = cfg.artifacts_dir / "best_lstm_autoencoder.pt"
assert ckpt_path.exists(), f"Checkpoint not found: {ckpt_path}"
state = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(state["model_state"])
model.eval()

test_loss, test_acc = run_epoch(
    model,
    test_loader,
    criterion,
    pad_id,
    DEVICE,
    train=False,
    desc="test",
)
test_perplexity = math.exp(min(test_loss, 20.0))
print({
    "test_loss": test_loss,
    "test_token_accuracy": test_acc,
    "test_perplexity": test_perplexity,
})

x = next(iter(test_loader)).to(DEVICE)
with torch.no_grad():
    logits = model(x)
    preds = logits.argmax(dim=-1)

sample_idx = 0
true_tokens = x[sample_idx].cpu().tolist()
pred_tokens = preds[sample_idx].cpu().tolist()

true_trim = trim_pad(true_tokens, pad_id)
pred_trim = trim_pad(pred_tokens, pad_id)

print("Ground truth tokens (first 80):", true_trim[:80])
print("Reconstructed tokens (first 80):", pred_trim[:80])

with open(cfg.artifacts_dir / "training_history.json", "w", encoding="utf-8") as f:
    json.dump(history, f, indent=2)
print("Saved history ->", cfg.artifacts_dir / "training_history.json")

# Export Task 1 deliverables in dedicated, reusable locations.
deliverables_dir = cfg.artifacts_dir / "deliverables"
plots_dir = deliverables_dir / "plots"
midi_dir = deliverables_dir / "midi"
for d in (plots_dir, midi_dir):
    d.mkdir(parents=True, exist_ok=True)

# 1) Save reconstruction loss curve plot.
import matplotlib.pyplot as plt

epochs = [row["epoch"] for row in history]
train_losses = [row["train_loss"] for row in history]
val_losses = [row["val_loss"] for row in history]

plt.figure(figsize=(8, 5))
plt.plot(epochs, train_losses, marker="o", label="Train Reconstruction Loss")
plt.plot(epochs, val_losses, marker="o", label="Val Reconstruction Loss")
plt.xlabel("Epoch")
plt.ylabel("Cross-Entropy Reconstruction Loss")
plt.title("Task 1 Reconstruction Loss Curve")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

loss_curve_path = plots_dir / "reconstruction_loss_curve.png"
plt.savefig(loss_curve_path, dpi=160)
plt.close()
print("Saved reconstruction loss curve ->", loss_curve_path)

# 2) Save 5 reconstructed MIDI samples.
import pretty_midi

note_token_count = 128 * (len(duration_bins) + 1) * (len(velocity_bins) + 1)
stride = 128 * (len(duration_bins) + 1)
max_time_shift = float(time_shift_bins[-1]) if len(time_shift_bins) > 0 else 0.1

def decode_token_event(token_shifted: int):
    if token_shifted <= 0:
        return None
    token = int(token_shifted) - 1
    if token < 0 or token >= token_space:
        return None

    if token < note_token_count:
        v_bin = token // stride
        rem = token % stride
        d_bin = rem // 128
        pitch = rem % 128

        duration = float(duration_bins[d_bin]) if d_bin < len(duration_bins) else float(duration_bins[-1])
        velocity = int(velocity_bins[v_bin]) if v_bin < len(velocity_bins) else 127
        return ("note", pitch, duration, velocity)

    if len(time_shift_bins) > 0:
        t_bin = token - note_token_count
        delta_t = float(time_shift_bins[t_bin]) if t_bin < len(time_shift_bins) else max_time_shift
        return ("time_shift", delta_t)

    return None

with torch.no_grad():
    sample_batch = next(iter(test_loader)).to(DEVICE)
    sample_logits = model(sample_batch)
    sample_preds = sample_logits.argmax(dim=-1).cpu().tolist()

num_to_save = min(5, len(sample_preds))
midi_paths = []
for i in range(num_to_save):
    pred_seq = trim_pad(sample_preds[i], pad_id)
    pm = pretty_midi.PrettyMIDI(initial_tempo=120.0)
    inst = pretty_midi.Instrument(program=0, name="reconstructed")

    t = 0.0
    for tok in pred_seq:
        event = decode_token_event(tok)
        if event is None:
            continue

        if event[0] == "time_shift":
            t += float(event[1])
            continue

        _, pitch, duration, velocity = event
        end_t = t + max(0.05, duration)
        inst.notes.append(
            pretty_midi.Note(
                velocity=max(1, min(127, int(velocity))),
                pitch=max(0, min(127, int(pitch))),
                start=float(t),
                end=float(end_t),
            )
        )

    pm.instruments.append(inst)
    out_path = midi_dir / f"reconstruction_sample_{i + 1:02d}.mid"
    pm.write(str(out_path))
    midi_paths.append(str(out_path))

index_path = midi_dir / "midi_samples_index.json"
with open(index_path, "w", encoding="utf-8") as f:
    json.dump({"files": midi_paths}, f, indent=2)

print("Saved MIDI samples ->", midi_dir)
print("Saved MIDI index ->", index_path)

Task 2

In [ ]:
from __future__ import annotations

import json
import os
import random
from dataclasses import dataclass
from pathlib import Path
from typing import List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pretty_midi
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

In [ ]:
@dataclass
class Task2Config:
    project_root: Path = Path(os.environ.get("PROJECT_ROOT", Path.cwd())).resolve()
    if not (project_root / "data").exists() and (project_root.parent / "data").exists():
        project_root = project_root.parent
    processed_dir: Path = project_root / "data" / "processed"
    metadata_file: Path = processed_dir / "preprocessing_metadata.json"
    artifacts_dir: Path = project_root / "artifacts" / "task2_vae"

    seq_len: int = 256
    batch_size: int = 16
    num_workers: int = 0

    embedding_dim: int = 128
    hidden_dim: int = 256
    latent_dim: int = 64
    num_layers: int = 2
    dropout: float = 0.2

    lr: float = 1e-3
    weight_decay: float = 1e-5
    epochs: int = 5
    grad_clip: float = 1.0
    beta: float = 1e-3

    # Set to None for full-dataset training.
    max_train_examples: int | None = 40000
    max_val_examples: int | None = 5000
    max_test_examples: int | None = 5000


# Loads token sequences for a split, applies +1 token shift for PAD=0, and supports capped subset loading.
def load_split_token_sequences(
    split: str,
    max_examples: int | None,
    manifest_map: dict,
    processed_dir: Path,
    vocab_size: int,
) -> List[List[int]]:
    manifest = manifest_map.get(split, [])
    sequences: List[List[int]] = []

    for rel in tqdm(manifest, desc=f"Loading {split}", unit="chunk"):
        chunk_path = processed_dir / rel
        if not chunk_path.exists():
            continue
        with np.load(chunk_path, allow_pickle=True) as data:
            raw = data["token_sequences"].tolist()
            for seq in raw:
                if not seq:
                    continue
                shifted = [min(int(t) + 1, vocab_size - 1) for t in seq]
                sequences.append(shifted)
                if max_examples is not None and len(sequences) >= max_examples:
                    return sequences
    return sequences

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

cfg = Task2Config()
cfg.artifacts_dir.mkdir(parents=True, exist_ok=True)
assert cfg.metadata_file.exists(), f"Missing metadata: {cfg.metadata_file}"
print("Using metadata:", cfg.metadata_file)

with open(cfg.metadata_file, "r", encoding="utf-8") as f:
    meta = json.load(f)

duration_bins = tuple(meta["config"]["duration_bins"])
velocity_bins = tuple(meta["config"]["velocity_bins"])
time_shift_bins = tuple(meta["config"].get("time_shift_bins", []))

note_token_space = 128 * (len(duration_bins) + 1) * (len(velocity_bins) + 1)
time_shift_token_space = len(time_shift_bins)
token_space = note_token_space + time_shift_token_space
pad_id = 0
vocab_size = token_space + 1
print(
    {
        "note_token_space": note_token_space,
        "time_shift_token_space": time_shift_token_space,
        "token_space": token_space,
        "vocab_size": vocab_size,
    }
)

train_sequences = load_split_token_sequences(
    "train",
    cfg.max_train_examples,
    meta["chunk_manifests"],
    cfg.processed_dir,
    vocab_size,
)
val_sequences = load_split_token_sequences(
    "val",
    cfg.max_val_examples,
    meta["chunk_manifests"],
    cfg.processed_dir,
    vocab_size,
)
test_sequences = load_split_token_sequences(
    "test",
    cfg.max_test_examples,
    meta["chunk_manifests"],
    cfg.processed_dir,
    vocab_size,
)

print({
    "train": len(train_sequences),
    "val": len(val_sequences),
    "test": len(test_sequences),
})

In [ ]:
train_ds = TokenSequenceDataset(train_sequences, cfg.seq_len, pad_id)
val_ds = TokenSequenceDataset(val_sequences, cfg.seq_len, pad_id)
test_ds = TokenSequenceDataset(test_sequences, cfg.seq_len, pad_id)

train_loader = DataLoader(
    train_ds,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    val_ds,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    test_ds,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=torch.cuda.is_available(),
)

print("Batches:", {
    "train": len(train_loader),
    "val": len(val_loader),
    "test": len(test_loader),
})

In [ ]:
model = SequenceVAE(
    vocab_size=vocab_size,
    embedding_dim=cfg.embedding_dim,
    hidden_dim=cfg.hidden_dim,
    latent_dim=cfg.latent_dim,
    num_layers=cfg.num_layers,
    dropout=cfg.dropout,
    pad_id=pad_id,
).to(DEVICE)

criterion = nn.CrossEntropyLoss(ignore_index=pad_id, reduction="none")
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=cfg.lr,
    weight_decay=cfg.weight_decay,
    betas=(0.9, 0.999),
)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")

best_val_loss = float("inf")
history = []

for epoch in range(1, cfg.epochs + 1):
    train_loss, train_recon, train_kl, train_acc = run_epoch(
        model,
        train_loader,
        criterion,
        cfg.beta,
        pad_id,
        DEVICE,
        train=True,
        optimizer=optimizer,
        grad_clip=cfg.grad_clip,
        desc=f"train e{epoch}",
    )
    val_loss, val_recon, val_kl, val_acc = run_epoch(
        model,
        val_loader,
        criterion,
        cfg.beta,
        pad_id,
        DEVICE,
        train=False,
        desc=f"val e{epoch}",
    )

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "train_recon_loss": train_recon,
        "val_recon_loss": val_recon,
        "train_kl": train_kl,
        "val_kl": val_kl,
        "train_acc": train_acc,
        "val_acc": val_acc,
    }
    history.append(row)
    print(row)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        ckpt_path = cfg.artifacts_dir / "best_sequence_vae.pt"
        torch.save(
            {
                "model_state": model.state_dict(),
                "config": cfg.__dict__,
                "vocab_size": vocab_size,
                "pad_id": pad_id,
                "history": history,
            },
            ckpt_path,
        )
        print(f"Saved best checkpoint -> {ckpt_path}")

In [ ]:
# Load best checkpoint for final evaluation
ckpt_path = cfg.artifacts_dir / "best_sequence_vae.pt"
assert ckpt_path.exists(), f"Checkpoint not found: {ckpt_path}"
state = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(state["model_state"])
model.eval()

test_loss, test_recon, test_kl, test_acc = run_epoch(
    model,
    test_loader,
    criterion,
    cfg.beta,
    pad_id,
    DEVICE,
    train=False,
    desc="test",
)
print({
    "test_loss": test_loss,
    "test_recon_loss": test_recon,
    "test_kl": test_kl,
    "test_token_accuracy": test_acc,
})

# Qualitative reconstruction sample
x = next(iter(test_loader)).to(DEVICE)
with torch.no_grad():
    logits, _, _ = model(x)
    preds = logits.argmax(dim=-1)

sample_idx = 0
true_tokens = x[sample_idx].cpu().tolist()
pred_tokens = preds[sample_idx].cpu().tolist()

true_trim = trim_pad(true_tokens, pad_id)
pred_trim = trim_pad(pred_tokens, pad_id)

print("Ground truth tokens (first 80):", true_trim[:80])
print("Reconstructed tokens (first 80):", pred_trim[:80])

with open(cfg.artifacts_dir / "training_history.json", "w", encoding="utf-8") as f:
    json.dump(history, f, indent=2)
print("Saved history ->", cfg.artifacts_dir / "training_history.json")

# Export Task 2 deliverables in dedicated folder.
deliverables_dir = cfg.artifacts_dir / "deliverables"
plots_dir = deliverables_dir / "plots"
midi_dir = deliverables_dir / "midi"
reports_dir = deliverables_dir / "reports"
for d in (plots_dir, midi_dir, reports_dir):
    d.mkdir(parents=True, exist_ok=True)

# Loss curves
epochs = [row["epoch"] for row in history]
train_recon = [row["train_recon_loss"] for row in history]
val_recon = [row["val_recon_loss"] for row in history]
train_kl = [row["train_kl"] for row in history]
val_kl = [row["val_kl"] for row in history]

plt.figure(figsize=(8, 5))
plt.plot(epochs, train_recon, marker="o", label="Train Recon")
plt.plot(epochs, val_recon, marker="o", label="Val Recon")
plt.xlabel("Epoch")
plt.ylabel("Reconstruction Loss")
plt.title("Task 2 VAE Reconstruction Loss")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
recon_curve_path = plots_dir / "reconstruction_loss_curve.png"
plt.savefig(recon_curve_path, dpi=160)
plt.close()

plt.figure(figsize=(8, 5))
plt.plot(epochs, train_kl, marker="o", label="Train KL")
plt.plot(epochs, val_kl, marker="o", label="Val KL")
plt.xlabel("Epoch")
plt.ylabel("KL Loss")
plt.title("Task 2 VAE KL Loss")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
kl_curve_path = plots_dir / "kl_loss_curve.png"
plt.savefig(kl_curve_path, dpi=160)
plt.close()

print("Saved plots ->", plots_dir)

# Token decoder for MIDI export
note_stride = 128 * (len(duration_bins) + 1)
note_token_space = 128 * (len(duration_bins) + 1) * (len(velocity_bins) + 1)

def decode_shifted_token(token_shifted: int) -> tuple[str, tuple[int, float, int] | float] | None:
    if token_shifted <= 0:
        return None
    token = int(token_shifted) - 1
    if token < 0 or token >= token_space:
        return None

    if token < note_token_space:
        v_bin = token // note_stride
        rem = token % note_stride
        d_bin = rem // 128
        pitch = rem % 128

        duration = float(duration_bins[d_bin]) if d_bin < len(duration_bins) else float(duration_bins[-1])
        velocity = int(velocity_bins[v_bin]) if v_bin < len(velocity_bins) else 127
        return ("note", (pitch, duration, velocity))

    if time_shift_bins:
        ts_idx = token - note_token_space
        if 0 <= ts_idx < len(time_shift_bins):
            return ("time_shift", float(time_shift_bins[ts_idx]))

    return None


def tokens_to_midi(token_seq: List[int], out_path: Path) -> None:
    pm = pretty_midi.PrettyMIDI(initial_tempo=120.0)
    inst = pretty_midi.Instrument(program=0, name="vae_generated")
    t = 0.0
    for tok in token_seq:
        event = decode_shifted_token(tok)
        if event is None:
            continue
        event_type, payload = event
        if event_type == "time_shift":
            t += max(0.01, float(payload))
            continue

        pitch, duration, velocity = payload
        end_t = t + max(0.05, float(duration))
        inst.notes.append(
            pretty_midi.Note(
                velocity=max(1, min(127, int(velocity))),
                pitch=max(0, min(127, int(pitch))),
                start=float(t),
                end=float(end_t),
            )
        )
        t = end_t
    pm.instruments.append(inst)
    pm.write(str(out_path))


# 5 generated samples from prior (autoregressive)
num_generate = 5
generated_paths = []
with torch.no_grad():
    for i in range(num_generate):
        z = torch.randn(1, cfg.latent_dim, device=DEVICE)
        pred = model.generate(z, max_len=cfg.seq_len)[0].cpu().tolist()
        pred_trim = trim_pad(pred, pad_id)
        out_path = midi_dir / f"generated_sample_{i + 1:02d}.mid"
        tokens_to_midi(pred_trim, out_path)
        generated_paths.append(str(out_path))

index_path = midi_dir / "generated_midi_index.json"
with open(index_path, "w", encoding="utf-8") as f:
    json.dump({"files": generated_paths}, f, indent=2)

summary = {
    "checkpoint": str(ckpt_path),
    "history": str(cfg.artifacts_dir / "training_history.json"),
    "reconstruction_curve": str(recon_curve_path),
    "kl_curve": str(kl_curve_path),
    "generated_midi_index": str(index_path),
}
with open(reports_dir / "task2_deliverables_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Saved MIDI samples ->", midi_dir)
print("Saved summary ->", reports_dir / "task2_deliverables_summary.json")